# State-classification workflow (qubex-emulator)

> This is the qubex-emulator version of the upstream qubex experiment notebook. It uses synthetic `FakeExperiment` results and does not connect to hardware.

This notebook covers both binary and ternary readout classification.
Start with a standard `ge` experiment for the 2-state classifier, then
move to `ge-ef-cr` mode when you want to resolve `|g>`, `|e>`, and `|f>`.

## 1. Build a 2-state classifier

Start from the ordinary `ge` configuration and collect the readout
clusters for `|g>` and `|e>`.

In [1]:
import numpy as np

import qubex_emulator as qx

exp_ge = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 2. Connect to the setup

Connect before you collect any state clusters.

In [2]:
exp_ge.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-cr-cr', drag_hpi_puls

## 3. Check the waveform

Use a waveform check to confirm that the readout path is stable before classification.

In [3]:
waveform_result = exp_ge.check_waveform()

## 4. Prepare the `ge` pulse set

Measure a baseline Rabi oscillation, calibrate the half-pi pulse, and save the note before building the 2-state classifier.

In [4]:
rabi_result = exp_ge.obtain_rabi_params()
hpi_result = exp_ge.calibrate_hpi_pulse()
exp_ge.calib_note.save()

## 5. Measure the 2-state clusters

Collect the `|g>` and `|e>` readout distributions for the two-state classifier.

In [5]:
binary_distributions = exp_ge.measure_state_distribution(
    exp_ge.qubit_labels,
    n_states=2,
)

## 6. Build the 2-state classifier

Fit the classifier from the measured two-state clusters.

In [6]:
binary_classifier = exp_ge.build_classifier(
    exp_ge.qubit_labels,
    n_states=2,
    n_shots=10000,
)

## 7. Create an `Experiment` in `ge-ef-cr` mode

Open a second experiment in `ge-ef-cr` mode for three-state classification.
`ge-ef-cr` assigns control channels in the order `ge`, `ef`, and `cr`, so
it is the mode you want when you need direct access to the `|f>` manifold.
If the active AWG profile leaves a control port with only one channel, that
port resolves to `ge` only, so you cannot drive EF on that qubit.

In [7]:
exp_gef = qx.Experiment(
    system_id="YOUR_SYSTEM_ID",
    muxes=[0],
    configuration_mode="ge-ef-cr",
    # config_dir="/path/to/qubex-config/config",
    # params_dir="/path/to/qubex-config/params/YOUR_SYSTEM_ID",
)

## 8. Connect the `ge-ef-cr` session

Connect the second session before you push the `ge-ef-cr` layout to hardware.

In [8]:
exp_gef.connect()

FakeExperiment(name='fake-qubex-two-qubit-system', device_id='YOUR_SYSTEM_ID', qubit_labels=('Q00', 'Q01', 'Q02', 'Q03'), qubit_frequencies=(7.157231, 8.032295, 7.812112, 6.944337), qubit_anharmonicities=(-0.393715, -0.487412, -0.421337, -0.365884), readout_frequencies=(6.752, 6.903, 6.844, 6.711), coupling_strength=0.005, qubit_lifetime=(20.0, 20.0), qubit_lifetimes=None, hpi_duration=24.0, pi_duration=24.0, readout_duration=1000.0, rzx90_duration=None, cx_duration=None, single_qubit_fidelity=None, two_qubit_fidelity=None, readout_assignment_error=None, positions=((0.0, 0.0), (1.0, 0.0), (2.0, 0.0), (3.0, 0.0)), calibrated_at=None, metadata={'chip_id': None, 'system_id': 'YOUR_SYSTEM_ID', 'muxes': [0], 'calib_note_path': None, 'calibration_valid_days': None, 'extra_options': {}}, readout_pre_margin=0.0, readout_post_margin=0.0, config_path='', params_path='', property_dir=PosixPath('.'), classifier_dir=PosixPath('.'), classifier_type='gmm', configuration_mode='ge-ef-cr', drag_hpi_puls

## 9. Configure the `ge-ef-cr` layout

Call `configure()` so the control hardware applies the `ge-ef-cr` channel
layout before you prepare `|f>` and collect ternary clusters.

> [!CAUTION]
> `configure()` changes the state of the control instruments. On shared
> systems, it can affect other users who are using the same hardware. Also,
> if a control port has only one AWG channel, that port stays `ge`-only and
> EF drive is not available there.

In [9]:
exp_gef.configure()

## 10. Measure the EF Rabi oscillation

Use the EF Rabi scan to confirm that the `|f>` manifold can be prepared reliably.

In [10]:
ef_rabi_result = exp_gef.obtain_ef_rabi_params(
    exp_gef.qubit_labels,
    time_range=np.arange(0, 201, 4),
    n_shots=1024,
)

## 11. Calibrate the EF half-pi pulse

Calibrate the EF half-pi pulse before collecting the three-state clusters.

In [11]:
ef_hpi_result = exp_gef.calibrate_ef_hpi_pulse(
    exp_gef.qubit_labels,
    n_rotations=1,
)

## 12. Measure the 3-state clusters

Collect the `|g>`, `|e>`, and `|f>` distributions for ternary classification.

In [12]:
ternary_distributions = exp_gef.measure_state_distribution(
    exp_gef.qubit_labels,
    n_states=3,
)

## 13. Build the 3-state classifier

Fit the ternary classifier from the measured three-state clusters.

In [13]:
ternary_classifier = exp_gef.build_classifier(
    exp_gef.qubit_labels,
    n_states=3,
    n_shots=1000,
)